# PhysDrive · 15 pakette tam süreli RGB ile nabız eğitimi

**Kaggle ayarları** — Defteri Private tut. **Accelerator → None**, **Internet → On** seç. **Add Input** ile 15 paketin tamamını içeren özel veri girdilerini bağla. Kaggle'ın açtığı klasörler, orijinal adlarını taşıyan ZIP'ler veya ikisinin karışımı kullanılabilir. `paket_listesi.json` isteğe bağlıdır.

Önce girdi kontrolünü çalıştır. “283” göründüğünde eğitim hücresine geç. Program 48 katılımcıyı sabit tohumla 32 eğitim, 8 doğrulama ve 8 test kişisine ayırır. Bir kişinin bütün sürüşleri aynı bölümde kalır.

**Eğitim** — PURE üzerinde önceden eğitilmiş FactorizePhys modelinden başlanır. Her eğitim turunda uygun videoların tamamı 160 karelik kesitlere ayrılır, sondaki kareleri kapsamak için son kesit gerektiğinde öncekiyle örtüşür. Model RGB'den BVP dalgasını öğrenir. Doğrulama ve testte 10 saniyelik pencerelerdeki nabız tahmini aynı aralığın sensör kaydıyla (`HR.mat` medyanı) karşılaştırılır. Seçim ölçütü, oturumlara eşit ağırlık veren ortalama mutlak hatadır (BPM). Başlangıç modeli de doğrulamada ölçülür.

**Kapsam** — 30 fps ve kare indeksli etiketler beklenir. En fazla 30 karelik uzunluk farkında ortak süre kullanılır. Daha büyük farklar ve yüz tespit oranı %50 altındaki oturumlar rapora girer. Eğitimde düz veya sonlu olmayan BVP kesitleri, değerlendirmede yeterli geçerli HR örneği olmayan pencereler atlanır. Kaynaklar [PhysDrive](https://github.com/WJULYW/PhysDrive-Dataset) ve [rPPG-Toolbox](https://github.com/ubicomplab/rPPG-Toolbox).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/physdrive_full")
REPO = Path("/tmp/physdrive_full_code")
TOOLBOX = REPO / "external" / "rPPG-Toolbox"
PROJECT_VERSION = "f6e9b0d6bb491476b6806548c2f530630d063cd9"
TOOLBOX_VERSION = "b7500b848f84ad7f86e277b4612563b69f4f88f9"

def checkout(url, target, revision, paths):
    target.parent.mkdir(parents=True, exist_ok=True)
    if not (target / ".git").is_dir():
        subprocess.run(["git", "clone", "--depth=1", "--filter=blob:none", "--no-checkout",
                        url, str(target)], check=True)
    git = ["git", "-C", str(target)]
    if subprocess.run([*git, "cat-file", "-e", revision + "^{commit}"],
                      stderr=subprocess.DEVNULL).returncode:
        subprocess.run([*git, "fetch", "--depth=1", "origin", revision], check=True)
    subprocess.run([*git, "sparse-checkout", "set", "--no-cone", *paths], check=True)
    subprocess.run([*git, "checkout", "--detach", revision], check=True)

checkout("https://github.com/Nisagwn/rppg.git", REPO, PROJECT_VERSION, ["/scripts/", "/rppg/"])
checkout("https://github.com/ubicomplab/rPPG-Toolbox.git", TOOLBOX, TOOLBOX_VERSION,
         ["/neural_methods/", "/final_model_release/PURE_FactorizePhys_FSAM_Res.pth"])
RUN = [sys.executable, "-u", str(REPO / "scripts/physdrive_full.py"), "--input", str(INPUT)]


In [ ]:
# Sadece doğrulama. ZIP'ler açılmadan dizinleri okunur.
subprocess.run([*RUN, "--inventory-only"], check=True)


### Uzun CPU koşusuna devam etme

**Save Version → Save & Run All** ile çıktıları sakla. Aşağıdaki süre her Kaggle koşusu için ayrılan çalışma süresidir. Defter son işlediği oturumdan sonra durur ve `physdrive_full_resume.pth` yazar. Bir oturumun tamamlanması süreyi biraz aşabilir.

Yeni koşuda 15 veri paketi bağlı kalsın. Önceki sürümün **Notebook Output** girdisini de **Add Input** ile ekle. Tek bir devam dosyası varsa defter onu seçer. Birden çok dosya bulursa aşağıdaki `RESUME_FILE` değişkenine istediğin dosyanın tam yolunu gir. Devam ederken epoch ve batch ayarlarını koru.


In [ ]:
EPOCHS = 3
BATCH_SIZE = 2
HOURS_PER_RUN = 9
RESUME_FILE = ""  # İsteğe bağlı: /kaggle/input/.../physdrive_full_resume.pth

resume = None
if not (OUT / "physdrive_full_resume.pth").is_file():
    if RESUME_FILE:
        resume = Path(RESUME_FILE)
        if not resume.is_file():
            raise FileNotFoundError(resume)
    else:
        candidates = sorted({p.resolve() for p in INPUT.rglob("physdrive_full_resume.pth")})
        if len(candidates) > 1:
            raise ValueError(f"RESUME_FILE ile bir dosya seçin: {candidates}")
        resume = next(iter(candidates), None)
args = [*RUN, "--out", str(OUT), "--epochs", str(EPOCHS), "--batch", str(BATCH_SIZE),
        "--hours", str(HOURS_PER_RUN)]
if resume:
    args.extend(["--resume", str(resume)])
subprocess.run(args, check=True)


In [ ]:
import json
report = json.loads((OUT / "rapor.json").read_text(encoding="utf-8"))
print("Tamamlandı" if report["completed"] else "Devam dosyası hazır", OUT)
for score in report["scores"]:
    print(score)
excluded = {r["session"]: r["excluded"] for r in report["sessions"] if "excluded" in r}
print("Dışlanan oturumlar ve nedenleri:", excluded)


### Çıktılar

Kaggle **Output → physdrive_full** klasöründe `rapor.json` oturumları, dışlama nedenlerini, başlangıç ve epoch doğrulama hatalarını içerir. Test hatası, en iyi doğrulama ağırlıklarıyla son aşamada eklenir. `best.pth` en düşük doğrulama hatasını veren ağırlıklardır. `last.pth` tamamlanan son eğitim turuna aittir. Devam etmek için model, optimizer ve sıradaki oturumu içeren `physdrive_full_resume.pth` kullanılır.
